# 1:

In [0]:
import re
from datetime import datetime, timezone

from pyspark.sql import functions as F

CATALOG = "workspace"      # change only if your catalog has another name

ROOT = f"/Volumes/{CATALOG}/landing/staging/ibm"
NEW_DIR = f"{ROOT}/new"
DONE_DIR = f"{ROOT}/processed"
BRONZE_TABLE = f"{CATALOG}.bronze.ibm_calibration_raw"

spark.conf.set("spark.sql.session.timeZone", "UTC")

In [0]:
from pyspark.sql.types import StructType, StructField, StringType

# Forced schema: exact CSV header names, in exact CSV order (23 columns).
# Bronze keeps raw values, so every column is StringType.
# Silver is where the data types get fixed.
IBM_COLUMNS = [
    "Qubit",
    "T1 (us)",
    "T2 (us)",
    "Readout assignment error",
    "Init error",
    "Prob meas0 prep1",
    "Prob meas1 prep0",
    "Readout length (ns)",
    "ID error",
    "Single-qubit gate length (ns)",
    "RX error",
    "Z-axis rotation (rz) error",
    "√x (sx) error",
    "Pauli-X error",
    "XSLOW error",
    "CZ error",
    "Gate length (ns)",
    "RZZ error",
    "MEASURE error",
    "MEASURE_2 error",
    "Operational",
    "Backend_ID",
    "Calibration_Timestamp",
]

IBM_SCHEMA = StructType([StructField(name, StringType(), True) for name in IBM_COLUMNS])

# 2:

In [0]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {BRONZE_TABLE} (
        backend_id                STRING,
        collection_timestamp      TIMESTAMP,
        raw_calibration_timestamp STRING,
        raw_payload               STRING,
        source                    STRING,
        source_file               STRING
    )
    USING DELTA
""")

DataFrame[]

# 3:

In [0]:
already_loaded = {row["source_file"] for row in spark.table(BRONZE_TABLE).select("source_file").distinct().collect()}
dbutils.fs.mkdirs(DONE_DIR)

csv_files = sorted([f for f in dbutils.fs.ls(NEW_DIR) if f.name.endswith(".csv")], key=lambda f: f.name)
print("CSV files waiting:", len(csv_files))

for f in csv_files:
    if f.name in already_loaded:
        print("Already in Bronze, only moving:", f.name)
    else:
        # When did the collector fetch this file? It is written in the file name.
        match = re.search(r"(\d{8}T\d{6}Z)", f.name)
        if match:
            collected_at = datetime.strptime(match.group(1), "%Y%m%dT%H%M%SZ")
        else:
            collected_at = datetime.now(timezone.utc).replace(tzinfo=None)

        raw = (
            spark.read
            .schema(IBM_SCHEMA)                  # forced schema, no inference
            .option("header", True)
            .option("inferSchema", False)        # explicit: never guess types
            .option("enforceSchema", False)      # check the CSV header against IBM_SCHEMA, by position and name
            .option("mode", "FAILFAST")          # a malformed row stops the load instead of becoming NULL
            .option("encoding", "UTF-8")         # the header contains "√x (sx) error"
            .csv(f.path)
        )

        bronze = raw.select(
            F.col("Backend_ID").alias("backend_id"),
            F.lit(collected_at).cast("timestamp").alias("collection_timestamp"),
            F.col("Calibration_Timestamp").alias("raw_calibration_timestamp"),
            F.to_json(F.struct("*"), {"ignoreNullFields": "false"}).alias("raw_payload"),
            F.lit("ibm_quantum_platform").alias("source"),
            F.lit(f.name).alias("source_file"),
        )

        bronze.write.mode("append").saveAsTable(BRONZE_TABLE)
        print("Loaded:", f.name, "| rows:", raw.count())

    dbutils.fs.mv(f.path, f"{DONE_DIR}/{f.name}")

CSV files waiting: 1
Already in Bronze, only moving: ibm_incremental_20261009T134114Z.csv


# 4:

In [0]:
# Quick health check
display(spark.sql(f"""
    SELECT backend_id,
           COUNT(*)                                             AS rows_loaded,
           COUNT(DISTINCT raw_calibration_timestamp)            AS snapshots,
           MIN(raw_calibration_timestamp)                       AS oldest_calibration,
           MAX(raw_calibration_timestamp)                       AS newest_calibration
    FROM {BRONZE_TABLE}
    GROUP BY backend_id
    ORDER BY backend_id
"""))

backend_id,rows_loaded,snapshots,oldest_calibration,newest_calibration
ibm_fez,155532,997,2026-03-01T19:07:26Z,2026-10-09T12:47:38Z
ibm_kingston,149604,959,2026-02-24T17:38:26Z,2026-10-09T12:54:09Z
ibm_marrakesh,151476,971,2026-03-01T19:03:17Z,2026-10-09T13:05:26Z


In [0]:
%sql
SELECT * FROM workspace.bronze.ibm_calibration_raw LIMIT 5

backend_id,collection_timestamp,raw_calibration_timestamp,raw_payload,source,source_file
ibm_marrakesh,2026-10-09T13:41:14.000Z,2026-10-09T13:05:26Z,"{""Qubit"":""0"",""T1 (us)"":""298.4677469542208"",""T2 (us)"":""40.417557291805714"",""Readout assignment error"":""0.0072021484375"",""Init error"":null,""Prob meas0 prep1"":""0.01123046875"",""Prob meas1 prep0"":""0.003173828125"",""Readout length (ns)"":""2684"",""ID error"":""0.0006455898642028418"",""Single-qubit gate length (ns)"":""36"",""RX error"":""0.0006455898642028418"",""Z-axis rotation (rz) error"":""0"",""√x (sx) error"":""0.0006455898642028418"",""Pauli-X error"":""0.0006455898642028418"",""XSLOW error"":""0.0006455898642028418"",""CZ error"":""1:0.004136906335316631;1:0.004136906335316631"",""Gate length (ns)"":""1:68;1:68"",""RZZ error"":""1:0.008660695550463504;1:0.008660695550463504"",""MEASURE error"":""0.0072021484375"",""MEASURE_2 error"":""0.003173828125"",""Operational"":""Yes"",""Backend_ID"":""ibm_marrakesh"",""Calibration_Timestamp"":""2026-10-09T13:05:26Z""}",ibm_quantum_platform,ibm_incremental_20261009T134114Z.csv
ibm_marrakesh,2026-10-09T13:41:14.000Z,2026-10-09T13:05:26Z,"{""Qubit"":""1"",""T1 (us)"":""143.7375014188929"",""T2 (us)"":""112.53063391351886"",""Readout assignment error"":""0.0172119140625"",""Init error"":null,""Prob meas0 prep1"":""0.012451171875"",""Prob meas1 prep0"":""0.02197265625"",""Readout length (ns)"":""2684"",""ID error"":""0.0002563328957151688"",""Single-qubit gate length (ns)"":""36"",""RX error"":""0.0002563328957151688"",""Z-axis rotation (rz) error"":""0"",""√x (sx) error"":""0.0002563328957151688"",""Pauli-X error"":""0.0002563328957151688"",""XSLOW error"":""0.0002563328957151688"",""CZ error"":""2:0.0024132830383006842;2:0.0024132830383006842;0:0.004136906335316631;0:0.004136906335316631"",""Gate length (ns)"":""2:68;2:68;0:68;0:68"",""RZZ error"":""2:0.009003036462655906;2:0.009003036462655906;0:0.008660695550463504;0:0.008660695550463504"",""MEASURE error"":""0.0172119140625"",""MEASURE_2 error"":""0.02197265625"",""Operational"":""Yes"",""Backend_ID"":""ibm_marrakesh"",""Calibration_Timestamp"":""2026-10-09T13:05:26Z""}",ibm_quantum_platform,ibm_incremental_20261009T134114Z.csv
ibm_marrakesh,2026-10-09T13:41:14.000Z,2026-10-09T13:05:26Z,"{""Qubit"":""2"",""T1 (us)"":""269.99010715603293"",""T2 (us)"":""149.65614384196064"",""Readout assignment error"":""0.007568359375"",""Init error"":null,""Prob meas0 prep1"":""0.009521484375"",""Prob meas1 prep0"":""0.005615234375"",""Readout length (ns)"":""2684"",""ID error"":""0.00028532641877750136"",""Single-qubit gate length (ns)"":""36"",""RX error"":""0.00028532641877750136"",""Z-axis rotation (rz) error"":""0"",""√x (sx) error"":""0.00028532641877750136"",""Pauli-X error"":""0.00028532641877750136"",""XSLOW error"":""0.00028532641877750136"",""CZ error"":""1:0.0024132830383006842;1:0.0024132830383006842;3:0.0016712075490877554;3:0.0016712075490877554"",""Gate length (ns)"":""1:68;1:68;3:68;3:68"",""RZZ error"":""1:0.009003036462655906;1:0.009003036462655906;3:0.0028203505352521474;3:0.0028203505352521474"",""MEASURE error"":""0.007568359375"",""MEASURE_2 error"":""0.005615234375"",""Operational"":""Yes"",""Backend_ID"":""ibm_marrakesh"",""Calibration_Timestamp"":""2026-10-09T13:05:26Z""}",ibm_quantum_platform,ibm_incremental_20261009T134114Z.csv
ibm_marrakesh,2026-10-09T13:41:14.000Z,2026-10-09T13:05:26Z,"{""Qubit"":""3"",""T1 (us)"":""266.6772166950027"",""T2 (us)"":""210.0924632137837"",""Readout assignment error"":""0.00537109375"",""Init error"":null,""Prob meas0 prep1"":""0.006591796875"",""Prob meas1 prep0"":""0.004150390625"",""Readout length (ns)"":""2684"",""ID error"":""0.00029089080827296844"",""Single-qubit gate length (ns)"":""36"",""RX error"":""0.00029089080827296844"",""Z-axis rotation (rz) error"":""0"",""√x (sx) error"":""0.00029089080827296844"",""Pauli-X error"":""0.00029089080827296844"",""XSLOW error"":""0.00029089080827296844"",""CZ error"":""